In [1]:
import pandas as pd
import numpy as np

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option('display.max_columns', None)

In [2]:
# Cargar el archivo de datos
df = pd.read_excel('../data/ODEs_1año.xlsx')

print(f"Dimensiones originales: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nPrimeras columnas disponibles:")
print(df.columns.tolist())

Dimensiones originales: 136,999 filas × 41 columnas

Primeras columnas disponibles:
['ID_ODE', 'T_USUARIO_ALTA', 'T_USUARIO_ULT_MOD', 'ESTADO_ODE', 'FECHA_ALTA_ODE', 'CONSIGNATARIO', 'SOLICITANTE', 'C_ID_REMITO', 'FECHA_DESP', 'C_ITEM', 'ESPESOR_DESP', 'ANCHO_DESP', 'LARGO_DESP', 'PESO_NETO_DESP', 'PESO_BRUTO_DESP', 'C_POSICION_ORDEN', 'C_PEDIDO', 'MATERIAL_PADRE', 'PESO_MATERIAL_ENTRADA', 'ANCHO_PADRE', 'LARGO_PADRE', 'ESPESOR_PADRE', 'ID_MAT', 'MATERIAL', 'PESO_MATERIAL_SALIDA', 'ANCHO_HIJO', 'LARGO_HIJO', 'ESPESOR_HIJO', 'DESCRIPCION_EMBALAJE', 'C_OFA_ID', 'D_UBICACION', 'NUMERO_PARTE_HIJO', 'PIEZAS_POR_PAQ_HIJO', 'PESO_MAX_DESPACHO_HIJO', 'ID_RED_HIJO', 'ARMA_TARIMA', 'FECHA_ARMADO', 'USUARIO_ARMA', 'DESARMA_TARIMA', 'FECHA_DESARME', 'USUARIO_DESARMA']


In [3]:
# Eliminar cintas que no requirieron armado de tarima
df = df.dropna(subset=['ARMA_TARIMA'])

print(f"Filas después de filtrar sin ARMA_TARIMA: {len(df):,}")
print(f"Filas eliminadas: {136_999 - len(df):,}")

Filas después de filtrar sin ARMA_TARIMA: 27,743
Filas eliminadas: 109,256


In [4]:
# Variables relevantes para el proyecto
variables_relevantes = [
    # Identificación de la tarima y desarme (variables centrales)
    'ARMA_TARIMA',
    'DESARMA_TARIMA',
    
    # Restricciones de armado
    'MATERIAL_PADRE',
    'CONSIGNATARIO',
    'PESO_NETO_DESP',
    
    # Dimensiones para calcular diámetro externo
    'LARGO_HIJO',
    'ESPESOR_HIJO',
    
    # Especificaciones del producto
    'NUMERO_PARTE_HIJO',
    'PIEZAS_POR_PAQ_HIJO',
    'PESO_MAX_DESPACHO_HIJO',
    
    # Tipo de embalaje (restricciones OJO HORIZONTAL / OJO VERTICAL)
    'DESCRIPCION_EMBALAJE',
    
    # Planta (afecta límite de ancho en OJO HORIZONTAL)
    'D_UBICACION',
    
    # Ancho para verificar homogeneidad de NUMERO_PARTE_HIJO
    'ANCHO_HIJO',
]

df = df[variables_relevantes]

print(f"Variables conservadas: {len(df.columns)}")
print(f"Dimensiones del DataFrame limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Variables conservadas: 13
Dimensiones del DataFrame limpio: 27,743 filas × 13 columnas


0 - Es que la tarima no se desarmó

1 - La tarima se desarmó

In [5]:
# Transformar DESARMA_TARIMA a binaria: tiene valor → 1, faltante → 0
df['DESARMA_TARIMA'] = df['DESARMA_TARIMA'].notna().astype(int)

print("Distribución de DESARMA_TARIMA:")
print(df['DESARMA_TARIMA'].value_counts())
print(f"\nPorcentaje de tarimas desarmadas: {df['DESARMA_TARIMA'].mean()*100:.1f}%")

Distribución de DESARMA_TARIMA:
DESARMA_TARIMA
0    22284
1     5459
Name: count, dtype: int64

Porcentaje de tarimas desarmadas: 19.7%


In [6]:
# Reemplazar 'NOA' por NaN en las columnas de especificaciones
df['PIEZAS_POR_PAQ_HIJO'] = df['PIEZAS_POR_PAQ_HIJO'].replace('NOA', np.nan)
df['PESO_MAX_DESPACHO_HIJO'] = df['PESO_MAX_DESPACHO_HIJO'].replace('NOA', np.nan)

# Convertir a numérico ahora que NOA ya no interfiere
df['PIEZAS_POR_PAQ_HIJO'] = pd.to_numeric(df['PIEZAS_POR_PAQ_HIJO'], errors='coerce')
df['PESO_MAX_DESPACHO_HIJO'] = pd.to_numeric(df['PESO_MAX_DESPACHO_HIJO'], errors='coerce')

print("Valores faltantes por columna:")
print(df[['PIEZAS_POR_PAQ_HIJO', 'PESO_MAX_DESPACHO_HIJO']].isna().sum())
print(f"\nPorcentaje NOA en PIEZAS_POR_PAQ_HIJO:   {df['PIEZAS_POR_PAQ_HIJO'].isna().mean()*100:.1f}%")
print(f"Porcentaje NOA en PESO_MAX_DESPACHO_HIJO: {df['PESO_MAX_DESPACHO_HIJO'].isna().mean()*100:.1f}%")

Valores faltantes por columna:
PIEZAS_POR_PAQ_HIJO       24010
PESO_MAX_DESPACHO_HIJO    19982
dtype: int64

Porcentaje NOA en PIEZAS_POR_PAQ_HIJO:   86.5%
Porcentaje NOA en PESO_MAX_DESPACHO_HIJO: 72.0%


In [10]:
# Convertir dimensiones a numérico
df['LARGO_HIJO'] = pd.to_numeric(df['LARGO_HIJO'], errors='coerce')
df['ESPESOR_HIJO'] = pd.to_numeric(df['ESPESOR_HIJO'], errors='coerce')

# LARGO_HIJO está en metros — convertir a mm para la fórmula
LARGO_HIJO_MM = df['LARGO_HIJO'] * 1000
D_INTERNO = 508  # mm, constante confirmada por Ternium

# Calcular diámetro externo: D_externo = √(508² + 4 × LARGO_mm × ESPESOR_mm / π)
df['D_EXTERNO'] = np.sqrt(D_INTERNO**2 + (4 * LARGO_HIJO_MM * df['ESPESOR_HIJO']) / np.pi)

print("Estadísticas de D_EXTERNO (mm):")
print(df['D_EXTERNO'].describe().round(2))
print(f"\nValores nulos en D_EXTERNO: {df['D_EXTERNO'].isna().sum()}")

Estadísticas de D_EXTERNO (mm):
count    27731.00
mean      1315.58
std        278.46
min        508.00
25%       1079.96
50%       1283.77
75%       1618.89
max       4356.91
Name: D_EXTERNO, dtype: float64

Valores nulos en D_EXTERNO: 12


In [13]:
# Eliminar filas con D_EXTERNO nulo (LARGO_HIJO o ESPESOR_HIJO faltantes)
filas_antes = len(df)
df = df.dropna(subset=['D_EXTERNO'])

print(f"Filas eliminadas por D_EXTERNO nulo: {filas_antes - len(df)}")
print(f"Filas restantes: {len(df):,}")

Filas eliminadas por D_EXTERNO nulo: 12
Filas restantes: 27,731


In [14]:
# Guardar el DataFrame limpio en outputs/
df.to_csv('../outputs/01_datos_limpios.csv', index=False)

print("Archivo guardado: outputs/01_datos_limpios.csv")
print(f"Dimensiones finales: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nResumen de variables en el archivo limpio:")
print(df.dtypes)

Archivo guardado: outputs/01_datos_limpios.csv
Dimensiones finales: 27,731 filas × 14 columnas

Resumen de variables en el archivo limpio:
ARMA_TARIMA                   str
DESARMA_TARIMA              int64
MATERIAL_PADRE                str
CONSIGNATARIO                 str
PESO_NETO_DESP            float64
LARGO_HIJO                float64
ESPESOR_HIJO              float64
NUMERO_PARTE_HIJO             str
PIEZAS_POR_PAQ_HIJO       float64
PESO_MAX_DESPACHO_HIJO    float64
DESCRIPCION_EMBALAJE          str
D_UBICACION                   str
ANCHO_HIJO                float64
D_EXTERNO                 float64
dtype: object
